<a href="https://colab.research.google.com/github/SabinaZet/olx-nieruchomosci/blob/main/olx_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Źródła

- [RealPython Pythonic Data Cleaning With pandas and NumPy](https://realpython.com/python-data-cleaning-numpy-pandas/)

# Set up GitHub connection and localisation

In [2]:
!git clone https://github.com/SabinaZet/olx-nieruchomosci.git

Cloning into 'olx-nieruchomosci'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 125 (delta 76), reused 117 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 26.50 KiB | 8.83 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [3]:
#add your modules to Colab sys
import sys
sys.path.append("/content/olx-nieruchomosci")  # dodaj główny folder

Change working directory witch magic command (%) - ! doesn't work in Colab

In [4]:
%cd /content/olx-nieruchomosci

/content/olx-nieruchomosci


Update connection:

In [19]:
!git fetch origin

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 5 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 2.86 KiB | 1.43 MiB/s, done.
From https://github.com/SabinaZet/olx-nieruchomosci
   50d5c37..ea0be05  main       -> origin/main


Check for changes and update if needed:

In [20]:
!git status

On branch main
Your branch is behind 'origin/main' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)

nothing to commit, working tree clean


In [21]:
!git pull

Updating 50d5c37..ea0be05
Fast-forward
 main.py           |  22 +++--
 pipeline/clean.py | 269 +++++++++++++++++++++++++++++++++++++++++++++++++-----
 2 files changed, 263 insertions(+), 28 deletions(-)


# Normalizing Data Frames

## Get and pre-clean the data from OLX

In [7]:
from pipeline import paginate, normalized_data
import pipeline.clean as pc
import pandas as pd

#list of dicts with scraped pages data,metadata and links
pages = paginate()

#dict with data split into categories and DFs as values
data_categories = normalized_data(pages)

#print df.info() and df.head(2) for all dfs
#pc.key_info(data_categories)

#delete NaN columns and duplicate id from all dfs and check
dfs_cl = pc.clean_nan_df(data_categories)
dfs_cl = pc.deduplicate(dfs_cl)
pc.key_info(dfs_cl)

#print df name and each column with the number of unique values
pc.check_data(dfs_cl)

No 'next' link

data_location
<class 'pandas.core.frame.DataFrame'>
Index: 1279 entries, 0 to 1299
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               1279 non-null   int64  
 1   isGpsrAvailable                  1279 non-null   bool   
 2   location.city.id                 1279 non-null   int64  
 3   location.city.name               1279 non-null   object 
 4   location.city.normalized_name    1279 non-null   object 
 5   location.city._nodeId            52 non-null     object 
 6   location.district.id             522 non-null    float64
 7   location.district.name           522 non-null    object 
 8   location.district._nodeId        19 non-null     object 
 9   location.region.id               1279 non-null   int64  
 10  location.region.name             1279 non-null   object 
 11  location.region.normalized_name  1279 non-null   object 


Delete unnecessary columns:

In [8]:
MAP = {
    'data_location' : 'map.zoom',
    'data_category' : 'offer_type',
    'data_business' : ['user.is_online','contact.courier'],
    'data_offer' : [
        'location.city.id', 'location.city._nodeId', 'location.city.normalized_name', 'location.district.id', 'location.district._nodeId',
        'location.region.id', 'location.region.normalized_name', 'location.region._nodeId'
        ]
}

for name, col in MAP.items():
  dfs_cl[name].drop(col, inplace=True, axis=1)

#print df name and each column with the number of unique values
pc.check_data(dfs_cl)


data_location
id                                 1279
isGpsrAvailable                       2
location.city.id                    475
location.city.name                  473
location.city.normalized_name       475
location.city._nodeId                42
location.district.id                141
location.district.name              131
location.district._nodeId            19
location.region.id                   16
location.region.name                 16
location.region.normalized_name      16
location.region._nodeId              12
map.lat                             816
map.lon                             816
map.radius                           13
map.show_detailed                     2
dtype: int64

data_timing
id                     1279
last_refresh_time      1196
created_time           1269
omnibus_pushup_time     598
valid_to_time          1250
dtype: int64

data_category
id                  1279
category.id           19
category.type          1
category._nodeId      11
dtype: int6

In [9]:
#change index to 'id'
for name, col in dfs_cl.items():
  dfs_cl[name] = dfs_cl[name].set_index('id')

pc.key_info(dfs_cl)


data_location
<class 'pandas.core.frame.DataFrame'>
Index: 1279 entries, 1050124482 to 1053915738
Data columns (total 16 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   isGpsrAvailable                  1279 non-null   bool   
 1   location.city.id                 1279 non-null   int64  
 2   location.city.name               1279 non-null   object 
 3   location.city.normalized_name    1279 non-null   object 
 4   location.city._nodeId            52 non-null     object 
 5   location.district.id             522 non-null    float64
 6   location.district.name           522 non-null    object 
 7   location.district._nodeId        19 non-null     object 
 8   location.region.id               1279 non-null   int64  
 9   location.region.name             1279 non-null   object 
 10  location.region.normalized_name  1279 non-null   object 
 11  location.region._nodeId          52 non-null     object 


# View each df, check dtype, NaN values, etc

In [46]:
dfs_cl.keys()

dict_keys(['data_location', 'data_timing', 'data_category', 'data_business', 'data_offer', 'data_promoted', 'offer_photos', 'offer_parameters'])

In [82]:
dfs_cl['data_business']['user created']

,user created
id,
1053259578,2011-12-08T18:02:37+01:00
1053647970,2014-05-23T01:05:01+02:00
1053177976,2014-03-03T14:23:55+01:00
1050662711,2012-09-04T18:09:32+02:00
1023797479,2014-05-23T01:05:01+02:00
...,...
1039680054,2016-06-11T12:49:54+02:00
1037078764,2014-10-14T17:09:54+02:00
926801959,2023-10-28T13:24:20+02:00


In [10]:
#unifying column names
COL_MAP = {
    'data_location' : {
        name : name.strip().replace('location.', '').replace('.', ' ')
        if 'location.' in name else
        name.strip().replace('.', ' ')
        for name in dfs_cl['data_location'].columns
        },
    'data_timing' : {name : name.strip().replace('_', ' ') for name in dfs_cl['data_timing'].columns},
    'data_category' : {
        name : name.strip().replace('category.', '')
        if '.id' not in name else
        name.strip().replace('.', ' ')
        for name in dfs_cl['data_category']
        },
    'data_business' : {name : name.strip().replace('.', ' ') for name in dfs_cl['data_business'].columns},
    'data_offer' : {name : name.strip().replace('location.', '').replace('.', ' ') for name in dfs_cl['data_offer'].columns},
    'data_promoted' : {name : name.strip().replace('promotion.', '') for name in dfs_cl['data_promoted'].columns}
}

for name, col in COL_MAP.items():
  dfs_cl[name].rename(columns=col, inplace=True)

pc.key_info(dfs_cl)


data_location
<class 'pandas.core.frame.DataFrame'>
Index: 1279 entries, 1050124482 to 1053915738
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   isGpsrAvailable         1279 non-null   bool   
 1   city id                 1279 non-null   int64  
 2   city name               1279 non-null   object 
 3   city normalized_name    1279 non-null   object 
 4   city _nodeId            52 non-null     object 
 5   district id             522 non-null    float64
 6   district name           522 non-null    object 
 7   district _nodeId        19 non-null     object 
 8   region id               1279 non-null   int64  
 9   region name             1279 non-null   object 
 10  region normalized_name  1279 non-null   object 
 11  region _nodeId          52 non-null     object 
 12  map lat                 1279 non-null   float64
 13  map lon                 1279 non-null   float64
 14  map radius     

In [65]:
dfs_cl.keys()

dict_keys(['data_location', 'data_timing', 'data_category', 'data_business', 'data_offer', 'data_promoted', 'offer_photos', 'offer_parameters'])

In [93]:
dfs_cl['data_business'].dtypes

,0
business,bool
protect_phone,object
contact chat,bool
contact name,object
contact negotiation,bool
contact phone,bool
shop subdomain,object
user id,int64
user uuid,object
user _nodeId,object


In [11]:
#normalizing dtypes in DataFrames
from pandas.api.types import is_datetime64_any_dtype, is_object_dtype, is_bool_dtype, is_numeric_dtype

DTYPE_MAP = {
    'data_location' : {
        'isGpsrAvailable' : 'bool',
        'city id' : 'object',
        'city name' : 'object',
        'city normalized_name' : 'object',
        'city _nodeId' : 'object',
        'region id' : 'object',
        'region name' : 'object',
        'region normalized_name' : 'object',
        'region _nodeId' : 'object',
        'map lat' : 'float',
        'map lon' : 'float',
        'map radius' : 'int',
        'map show_detailed' : 'bool',
        'district id' : 'object',
        'district name' : 'object',
        'district _nodeId' : 'object'
    },
    'data_timing' : {
        'last refresh time' : 'datetime',
        'created time' : 'datetime',
        'omnibus pushup time' : 'datetime',
        'valid to time' : 'datetime'
    },
    'data_category' : {
        'category id' : 'category',
        'type' : 'category',
        '_nodeId' : 'object'
    },
    'data_business' : {
        'business' : 'bool',
        'protect_phone' : 'bool',
        'contact chat' : 'bool',
        'contact name' : 'object',
        'contact negotiation' : 'bool',
        'contact phone' : 'bool',
        'shop subdomain' : 'object',
        'user id' : 'object',
        'user uuid' : 'object',
        'user _nodeId' : 'object',
        'user about' : 'object',
        'user b2c_business_page' : 'bool',
        'user banner_desktop' : 'object',
        'user banner_mobile' : 'object',
        'user company_name' : 'object',
        'user created' : 'datetime',
        'user last_seen' : 'datetime',
        'user logo_ad_page' : 'object',
        'user name' : 'object',
        'user other_ads_enabled' : 'bool',
        'user photo' : 'object',
        'user social_network_account_type' : 'category',
        'user verification status' : 'object',
        'partner code' : 'object'
    },
    'data_offer' : {
        '_nodeId' : 'object',
        'title' : 'object',
        'status' : 'category',
        'url' : 'object',
        'description' : 'object',
        'external_url' : 'object',
        'city name' : 'object',
        'region name' :'object',
        'district name' : 'object'

    },
    'data_promoted' : {
        'highlighted' : 'bool',
        'top_ad' : 'bool',
        'options' : 'object',
        'premium_ad_page' : 'bool',
        'urgent' : 'bool',
        'b2c_ad_page' : 'bool'
    }
}

for name, df in DTYPE_MAP.items():
  if name not in dfs_cl:
    continue

  for col_name, dtype in df.items():
    if col_name not in dfs_cl[name].columns:
            continue

    if dtype == 'datetime' and not is_datetime64_any_dtype(dfs_cl[name][col_name]):
      dfs_cl[name][col_name] = pd.to_datetime(dfs_cl[name][col_name], utc=True, errors="coerce").dt.tz_convert('Europe/Warsaw')

    elif dtype == 'object' and not is_object_dtype(dfs_cl[name][col_name]):
      dfs_cl[name][col_name] = dfs_cl[name][col_name].astype('object')

    elif dtype == 'bool' and not is_bool_dtype(dfs_cl[name][col_name]):
      dfs_cl[name][col_name] = dfs_cl[name][col_name].astype('boolean')

    elif dtype in ('float', 'int') and not is_numeric_dtype(dfs_cl[name][col_name]):
      dfs_cl[name][col_name] = pd.to_numeric(dfs_cl[name][col_name], errors='coerce')

    elif dtype == 'category':
      dfs_cl[name][col_name] = dfs_cl[name][col_name].astype('category')

pc.key_info(dfs_cl)


data_location
<class 'pandas.core.frame.DataFrame'>
Index: 1279 entries, 1050124482 to 1053915738
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   isGpsrAvailable         1279 non-null   bool   
 1   city id                 1279 non-null   object 
 2   city name               1279 non-null   object 
 3   city normalized_name    1279 non-null   object 
 4   city _nodeId            52 non-null     object 
 5   district id             522 non-null    object 
 6   district name           522 non-null    object 
 7   district _nodeId        19 non-null     object 
 8   region id               1279 non-null   object 
 9   region name             1279 non-null   object 
 10  region normalized_name  1279 non-null   object 
 11  region _nodeId          52 non-null     object 
 12  map lat                 1279 non-null   float64
 13  map lon                 1279 non-null   float64
 14  map radius     

In [12]:
#drop rows where is no url
dfs_cl['data_offer'].dropna(subset='url', inplace=True)